# Quick start: 3D bispectrum to 3PCF

This notebook follows the complete workflow $B_{3\mathrm{D}} + \mathrm{LOS} \rightarrow B_{2\mathrm{D}} \rightarrow \zeta$. The compact grids keep the tutorial quick and are not convergence recommendations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fastnc
from fastnc.bispectrum import SPTMatterBispectrum3D
from fastnc.multipole import NumericMultipoleConfig
from fastnc.projection import LOSProjector
from fastnc.threepcf import ThreePCF, ThreePCFConfig

fastnc.configure_logging("INFO")

## Physical inputs

The delta-like projector is an exact fixed-redshift evaluation rather than a narrow numerical window. It maps $k_i=\ell_i/\chi$ and applies neither LOS quadrature nor a $\chi^{-4}$ prefactor.

In [ ]:
b3d = SPTMatterBispectrum3D.simple_debug()
projector = LOSProjector.delta_like(z=0.5, chi=1300.0)
b2d = projector.project(b3d)

angular_value = b2d.evaluate(ell1=100.0, ell2=120.0, ell3=80.0)
print(angular_value)

## Target grid and configuration

Theta must be logarithmically spaced. Phi is the opening angle opposite vertex 1. Construction is lazy: creating ThreePCF does not yet evaluate multipoles or Hankel transforms.

In [ ]:
theta = np.geomspace(2.0e-3, 2.0e-2, 6)
phi = np.linspace(0.0, np.pi, 9)
config = ThreePCFConfig(
    spin=(0, 0, 0), Lmax=4, kmax=2,
    ell_min=20.0, ell_max=2000.0, n_ell=24,
    multipole=NumericMultipoleConfig(n_angle=65),
    use_coupling_cache=False,
)
threepcf = ThreePCF(config, b2d, theta, phi, route="numeric")
print(threepcf.calculation_plan())

## Evaluate the 3PCF

A scalar calculation has one independent epsilon component, (1, 1, 1). ZetaTable is passive data with component, $\theta_1$, $\theta_2$, and $\phi$ axes.

In [ ]:
zeta = threepcf.zeta()
scalar = zeta.get((1, 1, 1))
print(zeta.values.shape)

equilateral = scalar[np.arange(theta.size), np.arange(theta.size), :]
for index in (0, phi.size // 2, -1):
    plt.loglog(theta, np.abs(equilateral[:, index]), label=fr"$\phi={phi[index]:.2f}$")
plt.xlabel(r"$\theta$")
plt.ylabel(r"$|\zeta(\theta,\theta,\phi)|$")
plt.legend();

Calling threepcf.zeta() again reuses the in-memory result. Route selection belongs to ThreePCF rather than the bispectrum. Production work must test convergence in the grids and in Lmax and kmax.